# Online Courses Helper · 核心逻辑与工程演示

这是一个基于 Python + Playwright + Tkinter 的知网在线课程自动化工具。

本 notebook **只演示纯逻辑与数据匹配**：
- 不访问真实课程平台、不执行真实登录；
- 不含任何真实账号 / 密码 / cookie / token；
- 所有数据均为**明显虚构**；
- 直接调用仓库真实代码 `core`，不复制主程序实现；
- **请从仓库根目录打开**并 `Restart & Run All`。

In [1]:
import sys, os
print("Python:", sys.version.split()[0])
print("OS:", os.name)
repo_root = os.path.abspath(".")
assert os.path.exists(os.path.join(repo_root, "core.py")), "请从仓库根目录启动 Jupyter / 打开 demo.ipynb"
print("仓库根目录检查: OK")


Python: 3.11.5
OS: nt
仓库根目录检查: OK


In [2]:
import core
from core import (
    fmt_hms, parse_progress_items, parse_creds_text, parse_theme_url,
    is_course_complete, progress_for_course_safe, overall_progress,
    reached_cert_target, run_scoped_defaults, fallback_summary, has_full_creds, login_mode,
)
print("已导入真实 core 模块:", os.path.basename(core.__file__))


已导入真实 core 模块: core.py


## 1) 服务端 progress 标准化
`parse_progress_items` 优先按 **`courseId`** 建立进度映射；当服务端记录缺少 `courseId` 时，使用 `name:<courseName>` 作为兼容 fallback，并对重复同名 fallback 做歧义保护。

In [3]:
items = [
    {"courseId": "course-001", "courseName": "示例课程A", "progress": 100, "learnState": 2,
     "duration": 6300, "learnDuration": 6300, "finishDate": "2026-01-01"},
    {"courseId": "course-002", "courseName": "示例课程B", "progress": 40, "learnState": 1,
     "duration": 8100, "learnDuration": 3240, "finishDate": None},
    {"courseName": "示例课程C", "progress": 50, "learnDuration": 3150, "duration": 6300},
]
pm = parse_progress_items(items)
print("标准化后的 key:", list(pm))
for k in pm:
    e = pm[k]
    print("  %-16s progress=%-5s learnDuration=%-5s duration=%-5s complete=%s" % (
        k, e["progress"], e["learnDuration"], e["duration"], is_course_complete(e)))
print("缺 courseId 的条目用名称兜底:", "name:示例课程C" in pm)


标准化后的 key: ['course-001', 'course-002', 'name:示例课程C']
  course-001       progress=100.0 learnDuration=6300  duration=6300.0 complete=True
  course-002       progress=40.0  learnDuration=3240  duration=8100.0 complete=False
  name:示例课程C       progress=50.0  learnDuration=3150  duration=6300.0 complete=False
缺 courseId 的条目用名称兜底: True


## 2) courseId 防止同名课程串课
两门课同名不同 ID，按 ID 可正确区分。

In [4]:
dup = [{"courseId": "101", "courseName": "同名课程", "progress": 20, "learnDuration": 1200},
       {"courseId": "102", "courseName": "同名课程", "progress": 80, "learnDuration": 4800}]
pm = parse_progress_items(dup)
print("key 按 ID:", list(pm))
print("101 进度:", progress_for_course_safe(pm, "101", "同名课程", False)["progress"])
print("102 进度:", progress_for_course_safe(pm, "102", "同名课程", False)["progress"])


key 按 ID: ['101', '102']
101 进度: 20.0
102 进度: 80.0


## 3) 名称 fallback 与歧义保护
**Case A**：进度缺 courseId，但名称在专题内唯一 → 允许按 `courseName` 兜底。
**Case B**：进度缺 courseId 且名称不唯一 → 禁止兜底，避免把同一份进度复制给多门课。

In [5]:
pmA = parse_progress_items([{"courseName": "唯一课", "progress": 50}])
print("Case A 名称唯一 -> 允许兜底:", progress_for_course_safe(pmA, None, "唯一课", True)["progress"])
print("Case B theme 两门同名 -> 阻断 (unique=False):",
      progress_for_course_safe(pmA, 1001, "唯一课", False), progress_for_course_safe(pmA, 1002, "唯一课", False))
pmB = parse_progress_items([{"courseName": "同名课", "progress": 10}, {"courseName": "同名课", "progress": 90}])
print("Case B 进度侧同名歧义 -> fallback_summary:", fallback_summary(pmB))
print("Case B 歧义后自动匹配结果(应为空):", progress_for_course_safe(pmB, None, "同名课", True))


Case A 名称唯一 -> 允许兜底: 50.0
Case B theme 两门同名 -> 阻断 (unique=False): {} {}
Case B 进度侧同名歧义 -> fallback_summary: ([], ['同名课'])
Case B 歧义后自动匹配结果(应为空): {}


## 4) display_name / match_name 区分
`display_name`（`tutorTitle` 优先，或 `courseName`）用于 GUI / 日志；
`match_name`（严格取 `courseName`）用于 progress 缺 `courseId` 时的兼容匹配。

In [6]:
theme_item = {"courseId": "course-001", "tutorTitle": "张老师：示例课程", "courseName": "示例课程"}
display = (theme_item.get("tutorTitle", "") or theme_item.get("courseName", "")).strip()
match = (theme_item.get("courseName", "") or "").strip()
print("display_name:", display, "-> GUI / 日志")
print("match_name:", match, "-> 与 progress.courseName 兼容匹配")
pm = parse_progress_items([{"courseName": "示例课程", "progress": 60}])
print("用 match_name 命中兜底:", progress_for_course_safe(pm, None, match, True)["progress"])
print("若错误拿 display 匹配 -> 未命中:", progress_for_course_safe(pm, None, display, True))


display_name: 张老师：示例课程 -> GUI / 日志
match_name: 示例课程 -> 与 progress.courseName 兼容匹配
用 match_name 命中兜底: 60.0
若错误拿 display 匹配 -> 未命中: {}


## 5) 总进度：时长加权，而非简单平均
短课程 100%、长课程 50%。简单平均 = 75%；时长加权偏向**低进度的长课程**，故更低，且被限制在 0~100%。

In [7]:
entries = [{"learn_sec": 4800, "target": 4800}, {"learn_sec": 6000, "target": 12000}]
w = overall_progress(entries)
print("时长加权总进度:", round(w, 1), "%")
print("简单平均:", (100 + 50) / 2, "%")
print("差别来源于长课程(低进度)在加权中占比更大。")


时长加权总进度: 64.3 %
简单平均: 75.0 %
差别来源于长课程(低进度)在加权中占比更大。


## 6) 达到累计学习目标判断
项目用 `learnDuration` 汇总作为证书累计学习时间，达到目标即正常停止：$CERT = 15 \times 3600$。

In [8]:
T = 15 * 3600
for s in (14 * 3600 + 59 * 60 + 59, 15 * 3600, 16 * 3600 + 20 * 60):
    print(fmt_hms(s), "-> reached:", reached_cert_target(s, T))


14:59:59 -> reached: False
15:00:00 -> reached: True
16:20:00 -> reached: True


## 7) 每次“开始学习”都应重置的运行期状态
下列字段不能跨 run 残留，否则 GUI 会显示上一轮的课程 / 时长 / 日志，或误报“已达到目标”。

In [9]:
d = run_scoped_defaults()
for k, v in d.items():
    print("  %-16s -> %r" % (k, v))


  stop_requested   -> False
  paused           -> False
  done             -> False
  cert_reached     -> False
  courses          -> []
  current_cid      -> None
  total_sec        -> 0.0
  log_lines        -> []


## 8) storage_state 登录流程说明
```
有效 storage_state -> headless 复用
        |
        v invalid / missing
headed 登录（有完整账号密码则自动填写；否则完全手动）
        |
        v 登录成功
保存 storage_state -> 之后默认 headless
```
账号密码只是**可选自动填写**；推荐首次手动登录后删除明文密码文件，只保留 `session/storage_state.json`。

## 9) 模块职责
- `core.py` —— 纯逻辑：数据解析、匹配、进度/目标计算、登录态判定（可单测）。
- `cnki_client.py` —— 知网平台 API 薄封装：课程列表、学习进度、认证头捕获。
- `browser_session.py` —— Playwright 生命周期、storage_state、headed/headless 切换。
- `刷网课.py` —— GUI 与业务流程编排（Worker/Runner）。
- `tests/` —— 纯逻辑单测；`.github/workflows/tests.yml` —— CI。

## 10) 测试与 CI
```bash
python -m unittest discover -s tests -v
python -m py_compile core.py browser_session.py cnki_client.py 刷网课.py
```
CI 在 Python 3.9–3.12 上执行：requirements 安装、compile check、import check、unit test（不运行真实浏览器、不访问知网）。